# ПЗ 7 — Распознавание через LLM (Qwen / Gemini)

Отправляем кадры в мультимодальную LLM, получаем структурированное описание сцены в формате JSON.

In [ ]:
!pip install openai google-generativeai -q

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

BASE_DRIVE  = '/content/drive/MyDrive/cv-frames'
VIDEO_DIR   = f'{BASE_DRIVE}/видио'
FRAMES_DIR  = f'{BASE_DRIVE}/кадры'
RESULTS_DIR = f'{BASE_DRIVE}/результаты'

for d in [VIDEO_DIR, FRAMES_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)
frames = sorted(f for f in os.listdir(FRAMES_DIR) if f.endswith('.jpg'))
print(f'кадров: {len(frames)}')

In [ ]:
# вставьте свои ключи
GEMINI_API_KEY = 'ВАШ_КЛЮЧ_GEMINI'   # https://aistudio.google.com/app/apikey
QWEN_API_KEY   = 'ВАШ_КЛЮЧ_QWEN'    # https://dashscope.console.aliyun.com

## Вариант A — Google Gemini

In [ ]:
import json
import google.generativeai as genai
from PIL import Image
from tqdm.notebook import tqdm

genai.configure(api_key=GEMINI_API_KEY)
gemini = genai.GenerativeModel('gemini-1.5-flash')

def describe_frame(image_path):
    img = Image.open(image_path)
    prompt = 'Опиши объекты на кадре. Верни JSON: {"objects": [...], "scene": "...", "mood": "..."}'
    resp = gemini.generate_content([img, prompt])
    try:
        text = resp.text.strip().lstrip('```json').rstrip('```')
        return json.loads(text)
    except Exception:
        return {'raw': resp.text}

results = []
for fname in tqdm(frames[:10], desc='gemini'):
    res = describe_frame(f'{FRAMES_DIR}/{fname}')
    res['frame'] = fname
    results.append(res)

with open(f'{RESULTS_DIR}/llm_detections.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f'обработано: {len(results)} кадров')

## Вариант B — Qwen VL

In [ ]:
import base64
from openai import OpenAI

client = OpenAI(
    api_key=QWEN_API_KEY,
    base_url='https://dashscope.aliyuncs.com/compatible-mode/v1',
)

def describe_frame_qwen(image_path):
    with open(image_path, 'rb') as f:
        b64 = base64.b64encode(f.read()).decode()
    resp = client.chat.completions.create(
        model='qwen-vl-max',
        messages=[{'role': 'user', 'content': [
            {'type': 'image_url', 'image_url': {'url': f'data:image/jpeg;base64,{b64}'}},
            {'type': 'text', 'text': 'Опиши объекты. JSON: {"objects": [...], "scene": "..."}'},
        ]}],
        max_tokens=512,
    )
    try:
        return json.loads(resp.choices[0].message.content)
    except Exception:
        return {'raw': resp.choices[0].message.content}

# results = [describe_frame_qwen(f'{FRAMES_DIR}/{f}') for f in frames[:10]]